# Single-Kernel **Power Trace** — CPU Colab Demo

Reconstruct a **time-varying (phase-resolved) power curve** for a *single* GPU kernel from the
EnergAIzer analytical predictor — **no GPU required** (runs on a free Colab **CPU** runtime).

### What this shows
`estimator.lookup(...)` gives you one **average** power number per kernel. This notebook goes deeper:
it splits the kernel into its execution **phases** (memory **load** -> **compute** -> **writeback**, repeated
per *wave*, plus a lower-power **tail** wave) and draws power vs. time as a **staircase**.

### Important: MODELED, not MEASURED
Real per-kernel power **cannot** be sampled — NVML samples every ~10-100 ms while a kernel lasts
micro-to-milliseconds. This staircase is the predictor's **structural estimate**. Its area
(power x time) equals **exactly** the energy the predictor already reports (energy-conserving).

## Setup - Step 1: clone the repo + install CPU dependencies

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/shubhamOjha1000/single_kernel_GPU_model.git"
ROOT = "/content/single_kernel_GPU_model"
PKG  = os.path.join(ROOT, "energaizer-ispass26-artifact-main")   # the `gee` package lives here

if not os.path.isdir(PKG):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], check=True)
print("Repo at:", PKG)

# CPU-only deps (no torch/transformers needed for inference from a query)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "pandas", "scipy", "cvxpy", "pyyaml",
                "opt-einsum", "tqdm", "scikit-learn", "gdown"], check=True)

# Make BOTH the framework (`gee`) and our helper (`power_trace.py`, at repo root) importable
for p in (PKG, ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)
print("Dependencies installed.")

## Setup - Step 2: download the pre-collected LUT database

EnergAIzer needs the offline database of measured kernel latency/power (the lookup tables the
analytical model is calibrated against). ~hundreds of MB; downloaded once from Google Drive.

In [ ]:
import os, glob, subprocess

DB_BASE = os.path.join(PKG, "database", "data")
os.makedirs(DB_BASE, exist_ok=True)
DB_FILE_ID = "1krvqRFDnaqrJUT06V2psIua0wQr6ETAE"

def find_luts():
    return glob.glob(os.path.join(DB_BASE, "**", "*.csv"), recursive=True)

if not find_luts():
    archive = os.path.join(DB_BASE, "precollected_database.tar.gz")
    try:
        import gdown
        gdown.download(id=DB_FILE_ID, output=archive, quiet=False)
    except Exception as e:
        print("gdown failed:", e)
    if os.path.exists(archive):
        subprocess.run(["tar", "-xzf", archive, "-C", DB_BASE], check=True)
        os.remove(archive)
    else:
        subprocess.run(["bash", os.path.join(PKG, "misc", "gdrive_download.sh"),
                        "https://drive.google.com/file/d/%s/view" % DB_FILE_ID],
                       cwd=DB_BASE, check=True)

luts = find_luts()
print("Found %d LUT csv files." % len(luts))
print("Manual link (if needed): https://drive.google.com/file/d/%s/view" % DB_FILE_ID)
assert luts, "LUT database not found - download failed. See the note above."

## Setup - Step 3: normalize LUT layout
The LUT config references CSVs by bare filename, resolved against `lut_folder_abs_path`. Symlink
every CSV so it sits directly in `database/data`.

In [ ]:
import os, glob

for csv in glob.glob(os.path.join(DB_BASE, "**", "*.csv"), recursive=True):
    dest = os.path.join(DB_BASE, os.path.basename(csv))
    if os.path.abspath(csv) != os.path.abspath(dest) and not os.path.exists(dest):
        os.symlink(os.path.abspath(csv), dest)

flat = sorted(f for f in os.listdir(DB_BASE) if f.endswith(".csv"))
print("%d CSVs directly in database/data" % len(flat))

## Build the estimator (non-DVFS)
We build the estimator with `dvfs_aware=False` so the power model uses the clean form
`P = p_dram*a_dram + p_l2*a_l2 + p_smem*a_smem + p_math*a_math + p_other*a_other + p_static`,
which is what the phase decomposition is built on. (`yz8` == NVIDIA A100-40GB-PCIE.)

In [ ]:
from gee import get_gee

A100_GPU   = os.path.join(PKG, "config", "gpu", "yz8.yaml")
LUT_CONFIG = os.path.join(PKG, "experiments_endtoend", "exp_config", "a100_lut_config.yaml")

estimator = get_gee(
    gpu_yaml_path=A100_GPU,
    lut_yaml_path=LUT_CONFIG,
    dvfs_aware=False,                 # clean non-DVFS power model for tracing
    lut_folder_abs_path=DB_BASE,
)
print("Estimator built.")

## Predict the phase-resolved power trace
One bf16 Tensor-Core GEMM `(B x M x K) @ (B x K x N)`. `predict_power_trace` returns the
per-phase `(label, duration_ms, power_W)` segments; `plot_power_trace` draws the staircase.

In [ ]:
from power_trace import predict_power_trace, print_trace, plot_power_trace

query = {
    "batch": 1,
    "dimM": 4096,
    "dimN": 4096,
    "dimK": 4096,
    "precM": "bf16",
    "precA": "bf16",
    "useTensorCore": True,
}
query_type = ("gemm",
              "tc" if query["useTensorCore"] else "cuda",
              "{}_{}".format(query["precM"], query["precA"]))

TARGET_FREQ = 900   # MHz (core/SM clock)

trace = predict_power_trace(estimator, query, query_type,
                            target_freq=TARGET_FREQ, max_plot_waves=6, verbose=True)

print_trace(trace)
fig = plot_power_trace(trace, show=True)

### Read the graph
- **Blue bands** = memory phases (global -> shared **load** and result **writeback**): DRAM/L2 busy,
  math idle -> **lower** power.
- **Red bands** = compute phase (the MMA loop): Tensor cores + shared memory hot -> **higher** power.
- The pattern **repeats per wave**; the final **tail wave** runs on fewer SMs -> a visible **dip**.
- **Dashed line** = the single average power that `lookup()` would have returned.

Try editing the shape: a **tall-skinny** GEMM (small M/N, large K) is more memory-bound (flatter,
lower); a **large square** GEMM is more compute-bound (taller compute plateaus).

## Always-on illustration (no LUT needed)
If you just want to *see the concept* without the database, this synthetic example uses
representative A100 bf16 activity factors. Clearly labelled SYNTHETIC - the real numbers come
from the cell above.

In [ ]:
from power_trace import demo_synthetic_trace, print_trace, plot_power_trace

syn = demo_synthetic_trace(M=4096, N=4096, K=4096, n_waves=4)
print_trace(syn)
fig = plot_power_trace(syn, title="SYNTHETIC illustration - phase-resolved power staircase", show=True)

## Recap
- The predictor normally collapses each kernel to **one average power**.
- `power_trace.py` intercepts the **per-phase activity factors** + fitted **per-unit power
  coefficients** and reapplies the power formula **per phase** -> a **staircase** of
  `(duration, power)` steps.
- It is **modeled, not measured**, but **energy-conserving**: the area under the staircase equals
  the energy `lookup()` reports.
- Knobs: `target_freq` (clock), `max_plot_waves` (how many waves to draw), and the GEMM shape.